# **1 — Install library**

In [ ]:

%pip install -q dash plotly paho-mqtt


# **2 — Subscribe MQTT dan menerima JSON**

In [ ]:

import json
import time
import threading
from datetime import datetime
from zoneinfo import ZoneInfo
from collections import deque

import requests
import paho.mqtt.client as mqtt
import plotly.graph_objects as go

from dash import Dash, html, dcc, Input, Output
from google.colab import output

# =========================================================
# 1. CONFIGURATION
# =========================================================
MQTT_BROKER = "broker.hivemq.com"
MQTT_PORT = 8000
MQTT_PATH = "/mqtt"

# Harus sama dengan topic pada program ESP32 di Wokwi
MQTT_TOPIC = "iot-workshop/ardi2026/group01/distance"

THRESHOLD = 50.0
MAX_POINTS = 100
PORT = 8050
WIB = ZoneInfo("Asia/Jakarta")

# =========================================================
# 2. DATA STORAGE
# =========================================================
data_lock = threading.Lock()

timestamps = deque(maxlen=MAX_POINTS)
distances = deque(maxlen=MAX_POINTS)

data = {
    "distance": None,
    "status": "WAITING",
    "mqtt": "Connecting",
    "messages": 0,
    "alerts": 0,
    "last_update": None,
    "payload": "--",
    "error": "--"
}

# =========================================================
# 3. MQTT CALLBACKS
# =========================================================
def on_connect(client, userdata, flags, reason_code, properties):
    if reason_code == 0:
        with data_lock:
            data["mqtt"] = "Connected"
            data["error"] = "--"

        print("MQTT connected!")
        client.subscribe(MQTT_TOPIC)
        print("Subscribed topic:", MQTT_TOPIC)
    else:
        with data_lock:
            data["mqtt"] = f"Connection failed: {reason_code}"
            data["error"] = str(reason_code)

        print("MQTT connection failed:", reason_code)


def on_disconnect(client, userdata, disconnect_flags,
                  reason_code, properties):
    with data_lock:
        data["mqtt"] = "Disconnected"

    print("MQTT disconnected:", reason_code)


def on_message(client, userdata, message):
    try:
        payload = message.payload.decode("utf-8").strip()
        obj = json.loads(payload)

        distance = float(obj["distance"])

        if not 0 <= distance <= 400:
            raise ValueError("Distance must be between 0 and 400 cm")

        # Tentukan status berdasarkan threshold
        status = "ALERT" if distance <= THRESHOLD else "NORMAL"

        # Simpan datetime dengan zona waktu Jakarta
        now = datetime.now(WIB)

        with data_lock:
            timestamps.append(now)
            distances.append(distance)

            data["distance"] = distance
            data["status"] = status
            data["messages"] += 1

            if status == "ALERT":
                data["alerts"] += 1

            data["last_update"] = now
            data["payload"] = payload
            data["error"] = "--"

        print(
            f"[{now.strftime('%H:%M:%S')} WIB] "
            f"Distance: {distance:.1f} cm | {status}"
        )

    except Exception as exc:
        with data_lock:
            data["error"] = f"Invalid payload: {exc}"

        print("Message processing error:", exc)


# =========================================================
# 4. START MQTT CLIENT
# =========================================================
mqtt_client = mqtt.Client(
    callback_api_version=mqtt.CallbackAPIVersion.VERSION2,
    client_id=f"colab_iot_{int(time.time())}",
    transport="websockets"
)

mqtt_client.ws_set_options(path=MQTT_PATH)
mqtt_client.on_connect = on_connect
mqtt_client.on_disconnect = on_disconnect
mqtt_client.on_message = on_message
mqtt_client.reconnect_delay_set(min_delay=1, max_delay=30)

try:
    mqtt_client.connect_async(
        MQTT_BROKER,
        MQTT_PORT,
        keepalive=60
    )
    mqtt_client.loop_start()
    print("Connecting to HiveMQ...")
except Exception as exc:
    with data_lock:
        data["mqtt"] = "Connection error"
        data["error"] = str(exc)

    print("MQTT setup error:", exc)


# =========================================================
# 5. DASHBOARD DESIGN
# =========================================================
app = Dash(__name__)
app.title = "Mini IoT Monitoring Dashboard"

CARD_STYLE = {
    "backgroundColor": "white",
    "padding": "18px",
    "borderRadius": "12px",
    "boxShadow": "0 2px 8px rgba(0,0,0,0.08)",
    "flex": "1",
    "minWidth": "150px",
    "textAlign": "center"
}

app.layout = html.Div(
    style={
        "fontFamily": "Arial, sans-serif",
        "backgroundColor": "#f2f4f7",
        "minHeight": "100vh",
        "padding": "22px",
        "color": "#243447"
    },
    children=[
        html.H1(
            "Mini IoT Monitoring Dashboard",
            style={"textAlign": "center"}
        ),

        html.P(
            "ESP32 + HC-SR04 | Wokwi | MQTT HiveMQ",
            style={"textAlign": "center"}
        ),

        html.H3(
            id="mqtt-status",
            style={"textAlign": "center"}
        ),

        # SUMMARY CARDS
        html.Div(
            style={
                "display": "flex",
                "flexWrap": "wrap",
                "gap": "15px",
                "margin": "22px 0"
            },
            children=[
                html.Div([
                    html.H3("Distance"),
                    html.Div(
                        id="distance-value",
                        style={
                            "fontSize": "32px",
                            "fontWeight": "bold"
                        }
                    )
                ], style=CARD_STYLE),

                html.Div([
                    html.H3("System Status"),
                    html.Div(
                        id="system-status",
                        style={
                            "fontSize": "28px",
                            "fontWeight": "bold"
                        }
                    ),
                    html.P(f"Threshold: {THRESHOLD:.0f} cm")
                ], style=CARD_STYLE),

                html.Div([
                    html.H3("LED"),
                    html.Div(id="led-status", style={"fontSize": "24px"}),
                    html.H3("Buzzer"),
                    html.Div(id="buzzer-status", style={"fontSize": "24px"})
                ], style=CARD_STYLE),

                html.Div([
                    html.H3("Messages Received"),
                    html.Div(
                        id="message-count",
                        style={
                            "fontSize": "32px",
                            "fontWeight": "bold"
                        }
                    )
                ], style=CARD_STYLE),

                html.Div([
                    html.H3("Alert Readings"),
                    html.Div(
                        id="alert-count",
                        style={
                            "fontSize": "32px",
                            "fontWeight": "bold"
                        }
                    )
                ], style=CARD_STYLE)
            ]
        ),

        # REAL-TIME CHART
        html.Div(
            style={
                "backgroundColor": "white",
                "padding": "12px",
                "borderRadius": "12px",
                "marginBottom": "18px"
            },
            children=[
                dcc.Graph(
                    id="distance-chart",
                    config={"displayModeBar": True},
                    animate=False
                )
            ]
        ),

        # DIAGNOSTICS
        html.Div(
            style={
                "backgroundColor": "white",
                "padding": "18px",
                "borderRadius": "12px",
                "overflowWrap": "anywhere"
            },
            children=[
                html.H3("System Diagnostics"),
                html.P(id="last-update"),
                html.P(id="topic"),
                html.P(id="last-payload"),
                html.P(id="error-message")
            ]
        ),

        dcc.Interval(
            id="refresh",
            interval=1000,
            n_intervals=0
        )
    ]
)


# =========================================================
# 6. UPDATE DASHBOARD EVERY SECOND
# =========================================================
@app.callback(
    Output("mqtt-status", "children"),
    Output("mqtt-status", "style"),
    Output("distance-value", "children"),
    Output("system-status", "children"),
    Output("system-status", "style"),
    Output("led-status", "children"),
    Output("buzzer-status", "children"),
    Output("message-count", "children"),
    Output("alert-count", "children"),
    Output("distance-chart", "figure"),
    Output("last-update", "children"),
    Output("topic", "children"),
    Output("last-payload", "children"),
    Output("error-message", "children"),
    Input("refresh", "n_intervals")
)
def refresh_dashboard(_):
    with data_lock:
        current = data.copy()
        x = list(timestamps)
        y = list(distances)

    connected = current["mqtt"] == "Connected"
    alert = current["status"] == "ALERT"

    mqtt_style = {
        "textAlign": "center",
        "color": "#16803c" if connected else "#d92d20"
    }

    status_style = {
        "fontSize": "28px",
        "fontWeight": "bold",
        "color": "#d92d20" if alert else "#16803c"
    }

    distance_text = (
        f'{current["distance"]:.1f} cm'
        if current["distance"] is not None
        else "-- cm"
    )

    # -----------------------------------------------
    # CHART WITH CONNECTED LINE + MARKERS
    # -----------------------------------------------
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode="lines+markers",
        name="Distance",
        connectgaps=True,
        line={
            "color": "#5267ff",
            "width": 3,
            "shape": "linear"
        },
        marker={
            "size": 7,
            "color": "#5267ff"
        },
        hovertemplate=(
            "Time: %{x|%H:%M:%S} WIB<br>"
            "Distance: %{y:.1f} cm"
            "<extra></extra>"
        )
    ))

    # Threshold line
    fig.add_hline(
        y=THRESHOLD,
        line_dash="dash",
        line_width=2,
        annotation_text=f"Threshold {THRESHOLD:.0f} cm"
    )

    fig.update_layout(
        title="Real-Time Distance Monitoring",
        xaxis_title="Time (WIB)",
        yaxis_title="Distance (cm)",
        template="plotly_white",
        height=420,
        hovermode="x unified",
        margin={"l": 45, "r": 20, "t": 60, "b": 55},
        xaxis={
            "type": "date",
            "tickformat": "%H:%M:%S",
            "nticks": 8
        },
        yaxis={
            "rangemode": "tozero"
        }
    )

    # -----------------------------------------------
    # DIAGNOSTIC TEXT
    # -----------------------------------------------
    last_update = current["last_update"]

    last_update_text = (
        last_update.strftime("%Y-%m-%d %H:%M:%S WIB")
        if last_update is not None
        else "--"
    )

    diagnostic = current["error"]

    if current["messages"] == 0 and diagnostic == "--":
        diagnostic = (
            "No data received. Check Wokwi, MQTT topic, "
            "and MQTT connection."
        )

    return (
        f'MQTT Status: {current["mqtt"]}',
        mqtt_style,
        distance_text,
        current["status"],
        status_style,
        "🔴 ON" if alert else "🟢 OFF",
        "🔔 ON" if alert else "🔕 OFF",
        str(current["messages"]),
        str(current["alerts"]),
        fig,
        f"Last update: {last_update_text}",
        f"MQTT Topic: {MQTT_TOPIC}",
        f'Last payload: {current["payload"]}',
        f"Diagnostic: {diagnostic}"
    )


# =========================================================
# 7. START DASH SERVER
# =========================================================
def run_dashboard():
    app.run(
        host="0.0.0.0",
        port=PORT,
        debug=False,
        use_reloader=False
    )

dashboard_thread = threading.Thread(
    target=run_dashboard,
    daemon=True
)
dashboard_thread.start()

# Check server from inside the Colab runtime
server_ready = False

for attempt in range(20):
    try:
        response = requests.get(
            f"http://127.0.0.1:{PORT}",
            timeout=2
        )

        if response.status_code == 200:
            server_ready = True
            break

    except requests.RequestException:
        time.sleep(1)

if server_ready:
    print(f"Dashboard server active on port {PORT}")
    print("Opening dashboard through Google Colab proxy...")
    output.serve_kernel_port_as_window(PORT)
else:
    print("Dashboard server failed to start.")
    print("Check for port conflicts or errors above.")


Connecting to HiveMQ...


<IPython.core.display.Javascript object>

MQTT connected!
Subscribed topic: iot-workshop/ardi2026/group01/distance
Dashboard server active on port 8050
Opening dashboard through Google Colab proxy...
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>